In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-pptx', 'matplotlib', '--quiet', '--break-system-packages'], capture_output=True)

import os, re, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import BytesIO
from datetime import datetime
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

DASHBOARD_DIR = r"C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard"
DATA_DIR = os.path.join(DASHBOARD_DIR, "assets", "data")
OUTPUT   = os.path.join(DASHBOARD_DIR, "Informe_EcoGo_Clientes.pptx")

# Colors
TEAL_DARK  = RGBColor(0x1B, 0x5F, 0x5E)
TEAL       = RGBColor(0x3C, 0x97, 0x94)
TEAL_LIGHT = RGBColor(0x8F, 0xCC, 0xCA)
ORANGE     = RGBColor(0xFE, 0x8B, 0x5F)
RED        = RGBColor(0xDA, 0x45, 0x31)
WHITE      = RGBColor(0xFF, 0xFF, 0xFF)
BG         = RGBColor(0xF5, 0xF7, 0xF7)
TEXT       = RGBColor(0x1A, 0x25, 0x26)
MUTED      = RGBColor(0x6E, 0x76, 0x79)
GREEN      = RGBColor(0x1A, 0x7A, 0x45)

print("Setup OK")

In [ ]:
# --- Data loading ---
def read_data(filename):
    p = os.path.join(DATA_DIR, filename)
    if not os.path.exists(p):
        print(f"\u26a0 No encontrado: {filename}")
        return None
    txt = open(p, encoding='utf-8').read()
    # Remove single-line comments
    txt_clean = re.sub(r'//[^\n]*', '', txt)
    # Find first { and last }
    start = txt_clean.find('{')
    end   = txt_clean.rfind('}')
    if start == -1 or end == -1:
        print(f"\u26a0 No se pudo parsear: {filename}")
        return None
    try:
        return json.loads(txt_clean[start:end+1])
    except Exception as e:
        print(f"\u26a0 JSON error en {filename}: {e}")
        return None

# --- Format helpers ---
MESES_CORTO = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
MESES_LARGO = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

def fmt_pct(v, d=1):
    if v is None: return '\u2014'
    return f"{v*100:+.{d}f}%".replace('.', ',')

def fmt_pct_abs(v, d=1):
    if v is None: return '\u2014'
    return f"{v*100:.{d}f}%".replace('.', ',')

def fmt_mes(iso):
    if not iso: return ''
    m = re.match(r'(\d{4})-(\d{2})', str(iso))
    if not m: return str(iso)
    return MESES_CORTO[int(m.group(2))-1] + '-' + m.group(1)[2:]

def fmt_mes_largo(iso):
    if not iso: return ''
    m = re.match(r'(\d{4})-(\d{2})', str(iso))
    if not m: return str(iso)
    return MESES_LARGO[int(m.group(2))-1] + ' ' + m.group(1)

def fmt_num(v, d=1):
    if v is None: return '\u2014'
    return f"{v:,.{d}f}".replace(',', 'X').replace('.', ',').replace('X', '.')

def today_str():
    return datetime.now().strftime('%d de %B de %Y').lower().capitalize()

# --- Matplotlib style ---
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.grid': True,
    'grid.color': '#E0E4E4',
    'grid.linewidth': 0.7,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
})

def fig_to_bytes(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    buf.seek(0)
    plt.close(fig)
    return buf

# --- python-pptx helpers ---
def rgb(r, g, b):
    return RGBColor(r, g, b)

def add_rect(slide, x, y, w, h, fill_rgb, line_rgb=None, line_width=None):
    shape = slide.shapes.add_shape(1, Inches(x), Inches(y), Inches(w), Inches(h))
    shape.fill.solid()
    shape.fill.fore_color.rgb = fill_rgb
    if line_rgb:
        shape.line.color.rgb = line_rgb
        if line_width:
            shape.line.width = Pt(line_width)
    else:
        shape.line.fill.background()
    return shape

def add_text(slide, text, x, y, w, h, size=12, bold=False, color=None, align=PP_ALIGN.LEFT, italic=False, wrap=True):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tb.word_wrap = wrap
    p = tb.text_frame.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = str(text)
    run.font.size = Pt(size)
    run.font.bold = bold
    run.font.italic = italic
    run.font.color.rgb = color or TEXT
    return tb

def set_bg(slide, color_rgb):
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = color_rgb

def add_image_from_bytes(slide, buf, x, y, w, h):
    slide.shapes.add_picture(buf, Inches(x), Inches(y), Inches(w), Inches(h))

# --- Slide header ---
def slide_header(slide, title, subtitle=None):
    set_bg(slide, BG)
    add_rect(slide, 0, 0, 10, 0.52, TEAL_DARK)
    add_text(slide, 'ECO GO', 0.25, 0.04, 1.2, 0.44, size=10, bold=True, color=WHITE)
    add_text(slide, title, 1.4, 0.04, 6.5, 0.44, size=13, bold=True, color=WHITE)
    add_text(slide, today_str(), 7.5, 0.04, 2.3, 0.44, size=9, color=TEAL_LIGHT, align=PP_ALIGN.RIGHT)

# --- KPI card ---
def add_kpi(slide, x, y, w, h, label, value, sub=None, accent=None):
    accent = accent or TEAL_DARK
    add_rect(slide, x, y, w, h, WHITE, line_rgb=rgb(0xE0, 0xE4, 0xE4), line_width=0.75)
    add_rect(slide, x, y, 0.055, h, accent)
    add_text(slide, label, x+0.12, y+0.06, w-0.18, 0.2, size=8, color=MUTED, bold=True)
    add_text(slide, value, x+0.12, y+0.24, w-0.18, 0.38, size=20, bold=True, color=TEAL_DARK)
    if sub:
        add_text(slide, sub, x+0.12, y+0.6, w-0.18, 0.2, size=8, color=MUTED)

print("Helpers OK")

In [ ]:
PRECIOS  = read_data('precios.js')
EMAE     = read_data('emae_series.js')
RESERVAS = read_data('reservas.js')
TC       = read_data('tipo-cambio.js')
EMPLEO   = read_data('empleo.js')
MERCADOS = read_data('mercados.js')

# Validate
for name, d in [('Precios', PRECIOS), ('EMAE', EMAE), ('Reservas', RESERVAS)]:
    print(f"{'\u2705' if d else '\u274c'} {name}: {list(d.keys())[:5] if d else 'NO DATA'}")

In [ ]:
chart_images = {}

# ── RPM chart (últimos 36 meses) ──────────────────────────────────────────
if PRECIOS and PRECIOS.get('chart22'):
    c22 = [p for p in PRECIOS['chart22'] if p.get('fecha') and p.get('rpm_mm') is not None]
    c22 = c22[-36:]
    dates = [fmt_mes(p['fecha']) for p in c22]

    fig, ax1 = plt.subplots(figsize=(9, 3.5))
    ax2 = ax1.twinx()

    ax1.plot(range(len(dates)), [p.get('rpm_mm', 0)*100 for p in c22],
             color='#1B5F5E', lw=2.5, label='RPM Eco Go (m/m)')
    ax1.plot(range(len(dates)), [p.get('ipc_gba', 0)*100 if p.get('ipc_gba') else None for p in c22],
             color='#3C9794', lw=1.8, linestyle='--', label='IPC GBA (m/m)')
    ax1.plot(range(len(dates)), [p.get('ipc_nac', 0)*100 if p.get('ipc_nac') else None for p in c22],
             color='#FE8B5F', lw=1.8, linestyle=':', label='IPC Nac (m/m)')
    ax2.plot(range(len(dates)), [p.get('rpm_ia', 0)*100 if p.get('rpm_ia') else None for p in c22],
             color='#DA4531', lw=1.5, linestyle='-.', alpha=0.7, label='RPM i.a. (eje der.)')

    ticks = list(range(0, len(dates), max(1, len(dates)//8)))
    ax1.set_xticks(ticks)
    ax1.set_xticklabels([dates[i] for i in ticks], fontsize=8, rotation=35, ha='right')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}%'))
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax1.set_ylabel('Var. mensual', fontsize=8, color='#6E7679')
    ax2.set_ylabel('Var. i.a.', fontsize=8, color='#DA4531')
    ax2.tick_params(colors='#DA4531', labelsize=8)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, fontsize=7.5, loc='upper right', framealpha=0.9)
    ax1.set_title('RPM Eco Go vs IPC INDEC', fontsize=10, fontweight='bold', color='#1B5F5E', pad=8)
    fig.tight_layout()
    chart_images['rpm'] = fig_to_bytes(fig)
    print("\u2705 Chart RPM generado")

# ── EMAE original series (últimos 48 meses) ───────────────────────────────
if EMAE and EMAE.get('original'):
    orig = EMAE['original'][-48:]
    dates = [o.get('date', '') for o in orig]
    vals_o = [o.get('original') for o in orig]
    vals_d = [o.get('desest') for o in orig]

    # Compute i.a. (vs 12 months ago)
    ia = [None]*12 + [
        (orig[i]['original']/orig[i-12]['original']-1)*100
        if orig[i-12].get('original') else None
        for i in range(12, len(orig))
    ]

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax2 = ax.twinx()
    ax.plot(range(len(dates)), vals_o, color='#1B5F5E', lw=2, label='Serie original')
    ax.plot(range(len(dates)), vals_d, color='#3C9794', lw=1.8, linestyle='--', label='Desestacionalizada')
    ax2.plot(range(len(dates)), ia, color='#DA4531', lw=1.5, linestyle=':', alpha=0.8, label='Var. i.a. (eje der.)')

    ticks = list(range(0, len(dates), max(1, len(dates)//8)))
    ax.set_xticks(ticks)
    ax.set_xticklabels([fmt_mes(dates[i]) for i in ticks], fontsize=8, rotation=35, ha='right')
    ax.set_ylabel('\u00cdndice', fontsize=8, color='#6E7679')
    ax2.set_ylabel('Var. i.a. %', fontsize=8, color='#DA4531')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}%'))
    ax2.tick_params(colors='#DA4531', labelsize=8)
    lines1, lb1 = ax.get_legend_handles_labels()
    lines2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, lb1+lb2, fontsize=7.5, loc='upper left', framealpha=0.9)
    ax.set_title('EMAE \u00b7 Serie original y desestacionalizada', fontsize=10, fontweight='bold', color='#1B5F5E', pad=8)
    fig.tight_layout()
    chart_images['emae'] = fig_to_bytes(fig)
    print("\u2705 Chart EMAE generado")

# ── Reservas G5 (últimos 90 días) ─────────────────────────────────────────
if RESERVAS and RESERVAS.get('g5'):
    g5 = [p for p in RESERVAS['g5'] if p.get('r') is not None][-90:]
    dates = [p['d'] for p in g5]
    brutas = [p['r'] for p in g5]

    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.fill_between(range(len(dates)), brutas, alpha=0.15, color='#1B5F5E')
    ax.plot(range(len(dates)), brutas, color='#1B5F5E', lw=2)

    ticks = list(range(0, len(dates), max(1, len(dates)//6)))
    ax.set_xticks(ticks)
    ax.set_xticklabels([dates[i][:7] for i in ticks], fontsize=8, rotation=20, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'USD {v:,.0f}M'))
    ax.set_title('Reservas brutas \u00b7 BCRA (millones USD)', fontsize=9, fontweight='bold', color='#1B5F5E', pad=6)
    fig.tight_layout()
    chart_images['reservas'] = fig_to_bytes(fig)
    print("\u2705 Chart Reservas generado")

print(f"\nCharts generados: {list(chart_images.keys())}")

In [ ]:
prs = Presentation()
prs.slide_width  = Inches(10)
prs.slide_height = Inches(5.625)

blank_layout = prs.slide_layouts[6]  # blank

# ════════════════════════════════════════════════════════════════════
# SLIDE 1 — Portada
# ════════════════════════════════════════════════════════════════════
s = prs.slides.add_slide(blank_layout)
set_bg(s, TEAL_DARK)

# Bottom accent strip
add_rect(s, 0, 4.8, 10, 0.825, rgb(0x14, 0x4A, 0x49))

# Vertical accent line
add_rect(s, 0.7, 0.9, 0.07, 2.8, TEAL_LIGHT)

# Title
add_text(s, 'Informe Macroecon\u00f3mico', 0.95, 1.1, 8.5, 1.1, size=36, bold=True, color=WHITE)
add_text(s, 'Eco Go Consultores', 0.95, 2.15, 8.5, 0.6, size=20, color=TEAL_LIGHT)
add_text(s, today_str(), 0.95, 2.9, 8.5, 0.4, size=13, color=rgb(0xC0, 0xD8, 0xD7))

# Bottom text
add_text(s, 'Monitor Macroecon\u00f3mico  \u00b7  Uso confidencial', 0.3, 4.92, 8.5, 0.4,
         size=9, color=rgb(0x8F, 0xCC, 0xCA), align=PP_ALIGN.CENTER)
add_text(s, 'www.ecogo.com.ar', 8.0, 4.92, 1.8, 0.4, size=9, color=rgb(0x8F, 0xCC, 0xCA), align=PP_ALIGN.RIGHT)

print("\u2705 Slide 1 (Portada)")

# ════════════════════════════════════════════════════════════════════
# SLIDE 2 — Precios · IPC & RPM
# ════════════════════════════════════════════════════════════════════
s = prs.slides.add_slide(blank_layout)
slide_header(s, 'Precios \u00b7 IPC INDEC y RPM Eco Go')

if PRECIOS and PRECIOS.get('ultimo'):
    u = PRECIOS['ultimo']
    mes = fmt_mes_largo(u.get('fecha', ''))
    kw = 2.3; kh = 0.88; ky = 0.65; gap = 0.1; kx0 = 0.2

    kpis = [
        ('IPC General m/m',  fmt_pct_abs(u.get('general_mm')),   f"i.a.: {fmt_pct_abs(u.get('general_ia'))}",   TEAL_DARK),
        ('N\u00facleo m/m',  fmt_pct_abs(u.get('nucleo_mm')),    f"i.a.: {fmt_pct_abs(u.get('nucleo_ia'))}",    TEAL),
        ('Regulados m/m',    fmt_pct_abs(u.get('regulados_mm')), f"i.a.: {fmt_pct_abs(u.get('regulados_ia'))}", ORANGE),
        ('Estacionales m/m', fmt_pct_abs(u.get('estacional_mm')), f"i.a.: {fmt_pct_abs(u.get('estacional_ia'))}", rgb(0x7E, 0x5C, 0xCB)),
    ]
    for i, (label, val, sub, acc) in enumerate(kpis):
        add_kpi(s, kx0 + i*(kw+gap), ky, kw, kh, label, val, sub, acc)

    add_text(s, f'\u00daltimo dato: {mes}', 0.25, 0.6, 5, 0.22, size=8, color=MUTED)

# RPM Chart
if 'rpm' in chart_images:
    chart_images['rpm'].seek(0)
    add_image_from_bytes(s, chart_images['rpm'], 0.2, 1.62, 9.6, 3.75)
elif PRECIOS:
    add_text(s, 'Datos RPM no disponibles \u2014 ejecutar refresh primero', 0.5, 2.5, 9, 0.4,
             size=10, color=MUTED, align=PP_ALIGN.CENTER)

add_text(s, 'Fuente: IPC INDEC \u00b7 RPM Eco Go', 0.25, 5.42, 9, 0.2, size=7.5, color=MUTED)
print("\u2705 Slide 2 (Precios)")

# ════════════════════════════════════════════════════════════════════
# SLIDE 3 — RPM · Proyección
# ════════════════════════════════════════════════════════════════════
s = prs.slides.add_slide(blank_layout)
slide_header(s, 'RPM \u00b7 Proyecci\u00f3n pr\u00f3ximo mes')

if PRECIOS and PRECIOS.get('proyeccion') and PRECIOS['proyeccion'].get('filas'):
    proy = PRECIOS['proyeccion']
    filas = proy['filas']

    add_text(s, proy.get('header_periodo', 'Proyecci\u00f3n RPM'), 0.25, 0.6, 9.5, 0.28,
             size=10, bold=True, color=TEAL_DARK)

    headers = ['Cap\u00edtulo', 'Mensual', 'Acumulada', 'Anual (i.a.)']
    rows_data = [
        [f['capitulo'],
         fmt_pct_abs(f.get('mensual')),
         fmt_pct_abs(f.get('acumulada')),
         fmt_pct_abs(f.get('anual'))]
        for f in filas
    ]

    all_rows = [headers] + rows_data
    n_rows = len(all_rows)
    n_cols = 4

    tbl = s.shapes.add_table(n_rows, n_cols, Inches(0.2), Inches(0.95), Inches(9.6), Inches(4.35)).table
    tbl.columns[0].width = Inches(4.5)
    for ci in range(1, 4):
        tbl.columns[ci].width = Inches(1.7)

    for ri, row in enumerate(all_rows):
        for ci, val in enumerate(row):
            cell = tbl.cell(ri, ci)
            cell.text = str(val)
            p = cell.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.CENTER if ci > 0 else PP_ALIGN.LEFT
            run = p.runs[0] if p.runs else p.add_run()
            run.font.size = Pt(8.5 if ri > 0 else 9)
            is_general = ri > 0 and 'NIVEL GENERAL' in str(rows_data[ri-1][0]).upper()
            run.font.bold = (ri == 0 or is_general)

            if ri == 0:
                run.font.color.rgb = WHITE
                cell.fill.solid()
                cell.fill.fore_color.rgb = TEAL_DARK
            elif is_general:
                cell.fill.solid()
                cell.fill.fore_color.rgb = rgb(0xE8, 0xF4, 0xF4)
                run.font.color.rgb = TEAL_DARK
            else:
                cell.fill.solid()
                cell.fill.fore_color.rgb = WHITE if ri % 2 == 0 else rgb(0xF8, 0xFA, 0xFA)
                run.font.color.rgb = TEXT

add_text(s, 'Fuente: RPM Eco Go \u00b7 Cuadro Mensual por Cap\u00edtulos', 0.25, 5.42, 9, 0.2, size=7.5, color=MUTED)
print("\u2705 Slide 3 (RPM Proyecci\u00f3n)")

# ════════════════════════════════════════════════════════════════════
# SLIDE 4 — Actividad · EMAE
# ════════════════════════════════════════════════════════════════════
s = prs.slides.add_slide(blank_layout)
slide_header(s, 'Actividad Econ\u00f3mica \u00b7 EsAE Eco Go')

if EMAE and EMAE.get('original') and len(EMAE['original']) >= 13:
    orig = EMAE['original']
    last   = orig[-1]
    prev12 = orig[-13]
    prev1  = orig[-2]

    ia = (last['original']/prev12['original']-1)*100 if prev12.get('original') else None
    mm = (last['desest']/prev1['desest']-1)*100 if prev1.get('desest') else None

    def sign_color(v):
        if v is None: return TEAL_DARK
        return GREEN if v >= 0 else RED

    kw, kh = 4.6, 0.88
    add_kpi(s, 0.2, 0.65, kw, kh,
            f"EsAE \u00b7 Var. i.a. ({fmt_mes(last.get('date', ''))})",
            f"{ia:+.2f}%".replace('.', ',') if ia is not None else '\u2014',
            'vs. mismo mes a\u00f1o anterior', sign_color(ia))
    add_kpi(s, 5.0, 0.65, kw, kh,
            f"EsAE \u00b7 Var. m/m desest. ({fmt_mes(last.get('date', ''))})",
            f"{mm:+.2f}%".replace('.', ',') if mm is not None else '\u2014',
            'vs. mes anterior', sign_color(mm))

if 'emae' in chart_images:
    chart_images['emae'].seek(0)
    add_image_from_bytes(s, chart_images['emae'], 0.2, 1.65, 9.6, 3.72)

add_text(s, 'Fuente: INDEC / EsAE Eco Go \u00b7 Base 2004=100', 0.25, 5.42, 9, 0.2, size=7.5, color=MUTED)
print("\u2705 Slide 4 (Actividad)")

# ════════════════════════════════════════════════════════════════════
# SLIDE 5 — Reservas
# ════════════════════════════════════════════════════════════════════
s = prs.slides.add_slide(blank_layout)
slide_header(s, 'Reservas \u00b7 Banco Central de la Rep\u00fablica Argentina')

if RESERVAS:
    rin = RESERVAS.get('rin', {})
    g5  = RESERVAS.get('g5', [])

    # Last RIN date
    if rin.get('dates') and rin.get('rows'):
        last_date = rin['dates'][-1]
        add_text(s, f'\u00daltimo dato RIN: {last_date}', 0.25, 0.6, 5, 0.22, size=8, color=MUTED)

        # Find key rows
        kpi_rows = {}
        for row in rin['rows']:
            lbl = row.get('label', '').lower()
            val = row['vals'][-1] if row.get('vals') else None
            if 'brutas' in lbl:
                kpi_rows['brutas'] = val
            elif 'netas' in lbl:
                kpi_rows['netas'] = val
            elif 'l\u00edquidas' in lbl or 'liquidas' in lbl:
                kpi_rows['liquidas'] = val
            elif 'rin gob' in lbl or lbl.strip() == 'rin':
                kpi_rows['rin'] = val

        kw, kh = 2.2, 0.88
        kpis_r = [
            ('Reservas Brutas',   f"USD {fmt_num(kpi_rows.get('brutas'), 0)}M",   None, TEAL_DARK),
            ('Reservas L\u00edquidas', f"USD {fmt_num(kpi_rows.get('liquidas'), 0)}M", None, TEAL),
            ('Reservas Netas',    f"USD {fmt_num(kpi_rows.get('netas'), 0)}M",    None,
             rgb(0x7E, 0x5C, 0xCB) if (kpi_rows.get('netas') or 0) >= 0 else RED),
            ('RIN Gob. Nac.',     f"USD {fmt_num(kpi_rows.get('rin'), 0)}M",      None, ORANGE),
        ]
        for i, (label, val, sub, acc) in enumerate(kpis_r):
            add_kpi(s, 0.2 + i*(kw+0.1), 0.65, kw, kh, label, val, sub, acc)

    if 'reservas' in chart_images:
        chart_images['reservas'].seek(0)
        add_image_from_bytes(s, chart_images['reservas'], 0.2, 1.65, 9.6, 3.72)

add_text(s, 'Fuente: Banco Central de la Rep\u00fablica Argentina \u00b7 datos al \u00faltimo disponible',
         0.25, 5.42, 9, 0.2, size=7.5, color=MUTED)
print("\u2705 Slide 5 (Reservas)")

# ════════════════════════════════════════════════════════════════════
# SLIDE 6 — Cierre
# ════════════════════════════════════════════════════════════════════
s = prs.slides.add_slide(blank_layout)
set_bg(s, TEAL_DARK)
add_rect(s, 0, 4.8, 10, 0.825, rgb(0x14, 0x4A, 0x49))
add_rect(s, 0.7, 1.8, 0.07, 2.0, TEAL_LIGHT)

add_text(s, 'Eco Go Consultores', 0.95, 1.9, 8.5, 0.9, size=30, bold=True, color=WHITE)
add_text(s, 'Liderando el asesoramiento econ\u00f3mico desde 2005', 0.95, 2.75, 8.5, 0.5, size=14, color=TEAL_LIGHT)
add_text(s, 'www.ecogo.com.ar', 0.95, 3.3, 8.5, 0.4, size=12, color=rgb(0xC0, 0xD8, 0xD7))
add_text(s, 'Este informe es de uso confidencial y exclusivo para el destinatario. Eco Go no se responsabiliza por el uso de esta informaci\u00f3n por parte de terceros.',
         0.3, 4.9, 9.4, 0.4, size=7.5, color=rgb(0x8F, 0xCC, 0xCA), align=PP_ALIGN.CENTER)

print("\u2705 Slide 6 (Cierre)")

# Save
prs.save(OUTPUT)
print(f"\n\u2705 PPT guardado: {OUTPUT}")

In [ ]:
import subprocess
subprocess.Popen(['start', '', OUTPUT], shell=True)
print("Abriendo PowerPoint...")